# 04b — EXP_02: Naive RAG (BGE-large + ChromaDB top-k)

**Architecture:** dense retrieval + LLM. **Retriever:** `NaiveRetriever` (ChromaDB top-5 with the BGE-large query prefix). **Generator:** `llama-3.3-70b-versatile` via Groq · T=0 · max_tokens=700 (per Excel `Experiments Guide`). **Disk-cached** — restarts are free.

Same three-stage gating as EXP_01:

| Stage | Surface | Wall time | Cost | Gate |
|---|---|---|---|---|
| **A** Smoke | 50 stratified rows | ~1–2 min | $0 (Groq free tier) | always on |
| **B** Golden | 234 accepted golden rows | ~3–5 min | $0 | `RUN_GOLDEN = True` |
| **C** Test | **1,273 test-split rows** (locked 2026-05-06) | ~10 min | $0 | `RUN_FULL = False` (flip when A and B look right) |

Each stage writes `results/exp_02_naive_rag__<surface>/` with `predictions.jsonl`, `retrieval.jsonl` (now populated — top-5 chunk_ids + similarity scores per question), and `summary.json`. The runner is **resumable** — re-running any stage skips already-completed `question_id`s.

Tables this notebook fills: **Table 1** row 2 (`Acuuracy`, `Exact Match`, latency) · **Table 8** row 1 (Retrieval Quality — Recall@K, MRR, nDCG; computed against golden 234) · **Table 9** Naive-RAG column.

RAGAS metrics (all 5 — Faithfulness, Context Precision/Recall, Answer Relevancy, Answer Correctness) are filled by the companion notebook [`04b_exp02_ragas.ipynb`](04b_exp02_ragas.ipynb) after Stage B/C complete.

---

**The EXP_01 baseline to beat:** `Acuuracy = 0.7738` on the canonical test split (n=1,273) — the contamination-clean number that drove the 2026-05-06 surface narrowing. EXP_02's job is to demonstrate retrieval-grounded gain on this surface. Discussion-chapter hypothesis: *RAG raises test-split accuracy above 0.78 by surfacing chunks the LLM can ground on.* (For context: EXP_01's full-12,723 number was 0.8693, but that's contamination-inflated by train+dev memorisation — see `results/exp_01_base_llm__full_12723/README_LEGACY.md`.)

## 1. Setup

In [1]:
import json
import logging
import os
import sys
from pathlib import Path

# Quiet ChromaDB telemetry (matches the Notebook 03 pattern)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb").setLevel(logging.WARNING)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.data.embedder import best_device, load_bge
from src.data.indices import load_chroma
from src.data.loaders import load_golden, load_medqa_4opt
from src.eval.runner import run_experiment
from src.retrieval.naive import NaiveRetriever

print("GROQ_API_KEY present:", "\u2713" if os.environ.get("GROQ_API_KEY") else "\u2717")
print("Repo root:", REPO_ROOT)
print("Device   :", best_device())

GROQ_API_KEY present: ✓
Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project
Device   : mps


## 2. Build the retriever (one-time, ~3 s)

Loads BGE-large for query embeddings + opens the persistent ChromaDB collection built in Notebook 02. The model + collection are reused across all questions — don't re-create them per row.

In [2]:
CHROMA_DIR = REPO_ROOT / "data" / "indices" / "chroma_textbooks"

chroma_coll = load_chroma(CHROMA_DIR)
embedder = load_bge(device=best_device())
RETRIEVER = NaiveRetriever(chroma_coll, embedder)

print(f"ChromaDB count : {chroma_coll.count():,} chunks")
print(f"BGE device     : {embedder.device}")

# Sanity check on a known-good query
_test_chunks = RETRIEVER.retrieve(
    "What is the first-line treatment for community-acquired pneumonia?", k=3
)
print("\nSanity query (first-line CAP treatment) — top 3:")
for c in _test_chunks:
    print(f"  {c.chunk_id} (score={c.score:.4f}, book={c.book_name}): {c.text[:90]}…")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


ChromaDB count : 67,599 chunks
BGE device     : mps:0


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Sanity query (first-line CAP treatment) — top 3:
  InternalMed_Harrison_chunk_05120 (score=0.7217, book=InternalMed_Harrison): Management of bacteremic pneumococcal pneumonia also is controversial. Data from nonrandom…
  InternalMed_Harrison_chunk_05116 (score=0.7132, book=InternalMed_Harrison): Since the etiology of CAP is rarely known at the outset of treatment, initial therapy is u…
  Gynecology_Novak_chunk_00415 (score=0.7123, book=Gynecology_Novak): The American Thoracic Society updated their original guidelines in 2001 (10). These clinic…


## 3. Configuration

In [3]:
EXPERIMENT_ID = "EXP_02_NAIVE_RAG"
RESULTS_DIR = REPO_ROOT / "results"
TOP_K = 5  # locked by plan.md §0 #12

SMOKE_N = 50
SMOKE_SEED = 42

# Stage flags — flip these on once the previous stage looks right.
RUN_SMOKE = True       # Stage A · ~1–2 min
RUN_GOLDEN = True      # Stage B · ~3–5 min
RUN_FULL = True       # Stage C · ~10 min Groq on test split (1,273) — DON'T flip until A and B are clean

## 4. Stage A — Smoke (50 stratified rows)

Same stratified sample as EXP_01 (seed 42 + `meta_info` weighting), so the smoke surfaces are directly comparable across architectures. The first time you run this, expect ~3 s per row (BGE encode + Chroma top-5 + Groq); cache hits make re-runs free.

In [4]:
if RUN_SMOKE:
    medqa = load_medqa_4opt()
    smoke = (
        medqa.groupby("meta_info", group_keys=False)
        .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))
        .reset_index(drop=True)
    )
    smoke = smoke.head(SMOKE_N) if len(smoke) >= SMOKE_N else medqa.sample(n=SMOKE_N, random_state=SMOKE_SEED).reset_index(drop=True)

    print(f"Smoke surface: {len(smoke)} rows | meta_info mix: {dict(smoke['meta_info'].value_counts())}")

    smoke_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=smoke,
        output_dir=RESULTS_DIR / "exp_02_naive_rag__smoke_50",
        experiment_id=EXPERIMENT_ID,
        dataset_label="smoke_50",
        k=TOP_K,
    )
    print(json.dumps(smoke_summary, indent=2))
else:
    print("RUN_SMOKE = False — skipping Stage A")

Smoke surface: 50 rows | meta_info mix: {'step1': np.int64(28), 'step2&3': np.int64(22)}


/var/folders/q2/0276zgxs0m7bj83vqzpkxlqh0000gn/T/ipykernel_52447/1000236575.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))


EXP_02_NAIVE_RAG:   0%|          | 0/50 [00:00<?, ?it/s]

{
  "experiment_id": "EXP_02_NAIVE_RAG",
  "dataset": "smoke_50",
  "n_questions": 50,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.78,
  "Exact_Match": 0.78,
  "n_correct": 39,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.4531479644775391,
  "wall_time_s_this_run": 30.094808101654053,
  "n_calls_this_run": 50,
  "cache_hits_this_run": 0,
  "cache_hit_rate_this_run": 0.0,
  "timestamp_utc": "2026-05-06T17:07:07Z"
}


**Acceptance gate (Stage A):**
- `n_questions` == 50
- `Acuuracy` ≥ 0.70 (EXP_01 was 0.94 on this sample due to contamination; Naive RAG with retrieved context should land near 0.85–0.95)
- `mean_latency_s` < 8.0 (BGE encode + Chroma + Groq)
- `cache_hit_rate_this_run` is 0.0 on the first run, ~1.0 on a re-run
- Spot-check `retrieval.jsonl` — every row should have **5 `retrieved_chunk_ids`** with **non-zero `retrieved_chunk_scores`**

## 5. Stage B — Golden 234

Same retriever, golden surface. Cache will hit on rows that overlap with Stage A (rare unless the same question text appears in both).

In [5]:
if RUN_GOLDEN:
    golden = load_golden()
    print(f"Golden surface: {len(golden)} accepted rows")

    golden_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=golden,
        output_dir=RESULTS_DIR / "exp_02_naive_rag__golden_234",
        experiment_id=EXPERIMENT_ID,
        dataset_label="golden_234",
        k=TOP_K,
    )
    print(json.dumps(golden_summary, indent=2))
else:
    print("RUN_GOLDEN = False — skipping Stage B")

Golden surface: 234 accepted rows


EXP_02_NAIVE_RAG:   0%|          | 0/234 [00:00<?, ?it/s]

{
  "experiment_id": "EXP_02_NAIVE_RAG",
  "dataset": "golden_234",
  "n_questions": 234,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.8504273504273504,
  "Exact_Match": 0.8504273504273504,
  "n_correct": 199,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.4418208110026824,
  "wall_time_s_this_run": 132.77569103240967,
  "n_calls_this_run": 234,
  "cache_hits_this_run": 1,
  "cache_hit_rate_this_run": 0.004273504273504274,
  "timestamp_utc": "2026-05-06T17:14:55Z"
}


**Acceptance gate (Stage B):** `n_questions` == 234 · `Acuuracy` in roughly the smoke range. EXP_01 golden_234 was 0.902; expect EXP_02 to be in the 0.88–0.94 range — possibly a small lift, possibly a small loss (retrieval is a double-edged sword when the LLM already knows the answer from contamination). The interesting comparison is on the test-split-only slice, surfaced in the analysis cells below.

## 6. Stage C — Test split (1,273 rows · gated · ~10 min)

**Set `RUN_FULL = True` in the config cell only after Stages A and B look right.** This is the canonical headline run — evaluation surface narrowed to the MedQA US `test` split per [`plan.md` §0 #8](../plan.md) (locked 2026-05-06).

**Before flipping:** `caffeinate -dimsu &` to keep the Mac awake (optional for a 10-min run). The runner is resumable, so a mid-run failure (rate limit, network hiccup) just means re-running — already-done questions skip via `predictions.jsonl`.

**Note**: a partial full-12,723 run was started on 2026-05-06 and stopped at 4,844 rows (all train, no test rows touched) — abandoned under the new lock. Cache hits from that run will be zero on test_1273. See `results/exp_02_naive_rag__full_12723/README_LEGACY.md`.

In [4]:
if RUN_FULL:
    medqa = load_medqa_4opt()
    medqa = medqa[medqa["split"] == "test"].reset_index(drop=True)
    print(f"Test-split surface: {len(medqa)} rows")

    full_summary = run_experiment(
        retriever=RETRIEVER,
        dataset=medqa,
        output_dir=RESULTS_DIR / "exp_02_naive_rag__test_1273",
        experiment_id=EXPERIMENT_ID,
        dataset_label="test_1273",
        k=TOP_K,
    )
    print(json.dumps(full_summary, indent=2))
else:
    print("RUN_FULL = False — skipping Stage C (set RUN_FULL = True in the config cell when ready)")

Test-split surface: 1273 rows


EXP_02_NAIVE_RAG:   0%|          | 0/1273 [00:00<?, ?it/s]

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{
  "experiment_id": "EXP_02_NAIVE_RAG",
  "dataset": "test_1273",
  "n_questions": 1273,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.7572663000785546,
  "Exact_Match": 0.7572663000785546,
  "n_correct": 964,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.4436558864290847,
  "wall_time_s_this_run": 699.3165919780731,
  "n_calls_this_run": 1273,
  "cache_hits_this_run": 22,
  "cache_hit_rate_this_run": 0.01728201099764336,
  "timestamp_utc": "2026-05-06T18:33:46Z"
}


## 7. Inspect predictions — split-stratified comparison vs EXP_01

The headline thesis question is whether retrieval lifts accuracy on the **test split** (the contamination-clean slice). EXP_01 test-split was 0.774; EXP_02 should beat that if retrieval is doing real work.

In [5]:
import pandas as pd

for label in ("smoke_50", "golden_234", "test_1273"):
    p = RESULTS_DIR / f"exp_02_naive_rag__{label}" / "predictions.jsonl"
    if not p.exists():
        continue
    preds = pd.DataFrame(json.loads(line) for line in p.read_text().splitlines())
    print(f"\n=== {label}  (n={len(preds)}) ===")
    print(f"  overall accuracy : {preds['is_correct'].mean():.4f}")
    print(f"  mean latency     : {preds['latency_s'].mean():.3f} s")
    print(f"  cache hit rate   : {preds['was_cached'].mean()*100:.1f}%")

    if label == "test_1273":
        # Cross-join with medqa_4opt to get split column for the contamination breakdown
        medqa = load_medqa_4opt().reset_index(drop=True)
        joined = preds.merge(medqa[["question_id", "meta_info", "split"]], on="question_id", how="left")
        print("  by split:")
        for sp, g in joined.groupby("split"):
            print(f"    {sp:5s}: n={len(g):5d}  acc={g['is_correct'].mean():.4f}")
        print("  by meta_info:")
        for mi, g in joined.groupby("meta_info"):
            print(f"    {mi:10s}: n={len(g):5d}  acc={g['is_correct'].mean():.4f}")


=== smoke_50  (n=50) ===
  overall accuracy : 0.7800
  mean latency     : 0.453 s
  cache hit rate   : 0.0%

=== golden_234  (n=234) ===
  overall accuracy : 0.8504
  mean latency     : 0.442 s
  cache hit rate   : 0.4%

=== test_1273  (n=1273) ===
  overall accuracy : 0.7573
  mean latency     : 0.444 s
  cache hit rate   : 1.7%
  by split:
    test : n= 1273  acc=0.7573
  by meta_info:
    step1     : n=  679  acc=0.7585
    step2&3   : n=  594  acc=0.7559


## 8. Spot-check retrieval — do the chunks look on-topic?

Eyeball the first 3 questions and their retrieved chunks. If retrieval is healthy you should see (i) all 5 chunks per question, (ii) reasonable similarity scores (typically 0.5–0.8 for BGE-cosine), (iii) chunks that are topically related to the question.

In [6]:
smoke_ret_path = RESULTS_DIR / "exp_02_naive_rag__smoke_50" / "retrieval.jsonl"
smoke_pred_path = RESULTS_DIR / "exp_02_naive_rag__smoke_50" / "predictions.jsonl"

if smoke_ret_path.exists() and smoke_pred_path.exists():
    rets = pd.DataFrame(json.loads(line) for line in smoke_ret_path.read_text().splitlines())
    preds = pd.DataFrame(json.loads(line) for line in smoke_pred_path.read_text().splitlines())
    medqa = load_medqa_4opt()
    medqa_q_by_qid = dict(zip(medqa["question_id"], medqa["question"]))

    chunks_df = pd.read_parquet(REPO_ROOT / "data" / "processed" / "chunks.parquet")
    chunk_text_by_id = dict(zip(chunks_df.chunk_id, chunks_df.text))

    for i in range(min(3, len(rets))):
        ret_row = rets.iloc[i]
        pred_row = preds.iloc[i]
        qid = ret_row["question_id"]
        print(f"\n=== {qid}  pred={pred_row['pred_letter']}  gold={pred_row['gold_letter']}  correct={pred_row['is_correct']} ===")
        print(f"Q: {medqa_q_by_qid.get(qid, '(not joined)')[:160]}…")
        for cid, score in zip(ret_row["retrieved_chunk_ids"], ret_row["retrieved_chunk_scores"]):
            text = chunk_text_by_id.get(cid, "(missing)")
            print(f"  [{score:.4f}] {cid}: {text[:100].replace(chr(10), ' ')}…")
else:
    print("No smoke retrieval/prediction files yet — run Stage A first.")


=== medqa_09246  pred=D  gold=D  correct=True ===
Q: A 25-year-old woman first presented to your clinic due to morning stiffness, symmetrical arthralgia in her wrist joints, and fatigue. She had a blood pressure o…
  [0.7812] Pharmacology_Katzung_chunk_02062: Methotrexate, a synthetic nonbiologic antimetabolite, is the first-line csDMARD for treating RA and …
  [0.7370] Pharmacology_Katzung_chunk_03621: Allopurinol markedly reduces xanthine oxide catabolism of the purine analogs, potentially increasing…
  [0.7350] InternalMed_Harrison_chunk_11655: Methotrexate (MTX) inhibits dihydrofolate reductase, resulting in impaired DNA synthesis. Additional…
  [0.7261] Pharmacology_Katzung_chunk_03622: At higher dosage, methotrexate may cause bone marrow depression, megaloblastic anemia, alopecia, and…
  [0.7189] Pharmacology_Katzung_chunk_02063: 2.  Pharmacokinetics: Methotrexate can be administered either orally or parentally (SC or IM). The d…

=== medqa_12184  pred=C  gold=C  correct=True ==

---

**Next.** With EXP_02 baseline complete, run [`04b_exp02_ragas.ipynb`](04b_exp02_ragas.ipynb) to score the golden 234 on **all 5 RAGAS metrics** (Faithfulness, Context Precision, Context Recall, Answer Relevancy, Answer Correctness) using Claude Sonnet 4.6. Cost ~$30–40, wall time ~10–20 min. After that, the pattern repeats for EXP_03 (Sparse RAG / BM25), EXP_04 (Hybrid RAG), and EXP_05 (Multi-Hop RAG).